In [ ]:
from asn1crypto import x509, core, keys
import datetime

# Example 1: X.509 Certificate Parsing
print("Example 1: X.509 Certificate Operations")

try:
    # Create a proper RSA public key structure
    rsa_public_key = keys.RSAPublicKey({
        'modulus': core.Integer(12345678901234567890),
        'public_exponent': core.Integer(65537)
    })
    
    public_key_info = keys.PublicKeyInfo({
        'algorithm': {'algorithm': 'rsa'},
        'public_key': rsa_public_key
    })
    
    # Sample certificate data structure
    cert_info = x509.TbsCertificate({
        'version': 'v3',
        'serial_number': core.Integer(12345),
        'signature': {'algorithm': 'sha256_rsa'},
        'issuer': x509.Name.build({
            'common_name': 'Test CA'
        }),
        'subject': x509.Name.build({
            'common_name': 'Test Subject'
        }),
        'validity': {
            'not_before': core.UTCTime('240101000000Z'),
            'not_after': core.UTCTime('250101000000Z')
        },
        'subject_public_key_info': public_key_info
    })
    
    print("✓ Certificate structure created")
    print(f"Serial number: {cert_info['serial_number'].native}")
    print(f"Subject: {cert_info['subject'].human_friendly}")
    print(f"Issuer: {cert_info['issuer'].human_friendly}")
    print(f"Valid from: {cert_info['validity']['not_before'].native}")
    print(f"Valid to: {cert_info['validity']['not_after'].native}")
    
    # Test ASN.1 encoding
    encoded_cert = cert_info.dump()
    print(f"✓ ASN.1 encoded size: {len(encoded_cert)} bytes")
    
    # Test parsing back
    parsed_cert = x509.TbsCertificate.load(encoded_cert)
    print(f"✓ Certificate parsed back successfully")
    print(f"Parsed subject: {parsed_cert['subject'].human_friendly}")
    assert parsed_cert.dump() == encoded_cert  # DER re-encodes to the same bytes
    assert parsed_cert['serial_number'].native == 12345
    assert parsed_cert['version'].native == 'v3'
    assert parsed_cert['subject'].native == {'common_name': 'Test Subject'}
    assert parsed_cert['issuer'].native == {'common_name': 'Test CA'}
    utc = datetime.timezone.utc
    assert parsed_cert['validity']['not_before'].native == datetime.datetime(2024, 1, 1, tzinfo=utc)
    assert parsed_cert['validity']['not_after'].native == datetime.datetime(2025, 1, 1, tzinfo=utc)
    parsed_key = parsed_cert['subject_public_key_info']
    assert parsed_key.algorithm == 'rsa'
    assert parsed_key['public_key'].parsed['modulus'].native == 12345678901234567890
    
except Exception as e:
    print(f"Error in certificate parsing: {e}")
    raise AssertionError(
        f"asn1crypto example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
from asn1crypto import keys, core

# Example 2: Private Key Structure Operations
print("Example 2: Private Key Operations")

try:
    # Create RSA private key structure
    rsa_private_key = keys.RSAPrivateKey({
        'version': 0,
        'modulus': core.Integer(12345678901234567890),
        'public_exponent': core.Integer(65537),
        'private_exponent': core.Integer(98765432109876543210),
        'prime1': core.Integer(123456789),
        'prime2': core.Integer(987654321),
        'exponent1': core.Integer(1111),
        'exponent2': core.Integer(2222),
        'coefficient': core.Integer(3333)
    })
    
    print("✓ RSA private key structure created")
    print(f"Modulus: {rsa_private_key['modulus'].native}")
    print(f"Public exponent: {rsa_private_key['public_exponent'].native}")
    print(f"Version: {rsa_private_key['version'].native}")
    
    # Test encoding and decoding
    encoded_key = rsa_private_key.dump()
    print(f"✓ Private key encoded: {len(encoded_key)} bytes")
    
    # Parse back the key
    parsed_key = keys.RSAPrivateKey.load(encoded_key)
    print(f"✓ Private key parsed successfully")
    print(f"Parsed modulus matches: {parsed_key['modulus'].native == rsa_private_key['modulus'].native}")
    assert parsed_key.native == rsa_private_key.native
    assert parsed_key['private_exponent'].native == 98765432109876543210
    
    # Create PrivateKeyInfo wrapper
    key_info = keys.PrivateKeyInfo({
        'version': 0,
        'private_key_algorithm': {'algorithm': 'rsa'},
        'private_key': rsa_private_key
    })
    
    print("✓ PrivateKeyInfo wrapper created")
    print(f"Algorithm: {key_info['private_key_algorithm']['algorithm'].native}")
    reloaded_info = keys.PrivateKeyInfo.load(key_info.dump())
    assert reloaded_info.algorithm == 'rsa'
    assert reloaded_info['private_key'].parsed['modulus'].native == 12345678901234567890
    
except Exception as e:
    print(f"Error in private key operations: {e}")
    raise AssertionError(
        f"asn1crypto example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
from asn1crypto import csr, x509, core, keys

# Example 3: Certificate Signing Request Operations
print("Example 3: CSR Operations")

try:
    # Create a proper RSA public key structure
    rsa_public_key = keys.RSAPublicKey({
        'modulus': core.Integer(98765432109876543210),
        'public_exponent': core.Integer(65537)
    })
    
    public_key_info = keys.PublicKeyInfo({
        'algorithm': {'algorithm': 'rsa'},
        'public_key': rsa_public_key
    })
    
    # Create CSR info structure
    csr_info = csr.CertificationRequestInfo({
        'version': 0,
        'subject': x509.Name.build({
            'country_name': 'US',
            'state_or_province_name': 'California',
            'locality_name': 'San Francisco',
            'organization_name': 'Test Company',
            'common_name': 'test.example.com'
        }),
        'subject_pk_info': public_key_info,
        'attributes': []
    })
    
    print("✓ CSR info structure created")
    print(f"Subject: {csr_info['subject'].human_friendly}")
    print(f"Version: {csr_info['version'].native}")
    
    # Create full CSR
    cert_request = csr.CertificationRequest({
        'certification_request_info': csr_info,
        'signature_algorithm': {'algorithm': 'sha256_rsa'},
        'signature': core.OctetBitString(b'dummy_signature_data')
    })
    
    print("✓ Full CSR created")
    print(f"Signature algorithm: {cert_request['signature_algorithm']['algorithm'].native}")
    
    # Test encoding
    encoded_csr = cert_request.dump()
    print(f"✓ CSR encoded: {len(encoded_csr)} bytes")
    
    # Parse back
    parsed_csr = csr.CertificationRequest.load(encoded_csr)
    print("✓ CSR parsed successfully")
    
    # Extract subject from parsed CSR
    parsed_subject = parsed_csr['certification_request_info']['subject']
    print(f"Parsed subject: {parsed_subject.human_friendly}")
    assert parsed_subject.native == {
        'country_name': 'US',
        'state_or_province_name': 'California',
        'locality_name': 'San Francisco',
        'organization_name': 'Test Company',
        'common_name': 'test.example.com',
    }
    assert parsed_csr['signature'].native == b'dummy_signature_data'
    assert parsed_csr['signature_algorithm'].signature_algo == 'rsassa_pkcs1v15'
    assert parsed_csr['signature_algorithm'].hash_algo == 'sha256'
    parsed_pk = parsed_csr['certification_request_info']['subject_pk_info']
    assert parsed_pk['public_key'].parsed['modulus'].native == 98765432109876543210
    
except Exception as e:
    print(f"Error in CSR operations: {e}")
    raise AssertionError(
        f"asn1crypto example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

In [ ]:
from asn1crypto import cms, core, algos

# Example 4: PKCS#7/CMS (Cryptographic Message Syntax)
print("Example 4: PKCS#7/CMS Operations")

try:
    # Create signed data structure
    signed_data = cms.SignedData({
        'version': 1,
        'digest_algorithms': [
            algos.DigestAlgorithm({'algorithm': 'sha256'})
        ],
        'encap_content_info': {
            'content_type': 'data',
            'content': core.OctetString(b'This is the content to be signed')
        },
        'certificates': [],
        'signer_infos': []
    })
    
    print("✓ SignedData structure created")
    print(f"Version: {signed_data['version'].native}")
    print(f"Content type: {signed_data['encap_content_info']['content_type'].native}")
    print(f"Content: {signed_data['encap_content_info']['content'].native}")
    
    # Create enveloped data structure
    enveloped_data = cms.EnvelopedData({
        'version': 0,
        'recipient_infos': [],
        'encrypted_content_info': {
            'content_type': 'data',
            'content_encryption_algorithm': {
                'algorithm': 'aes128_cbc',
                'parameters': core.OctetString(b'1234567890123456')  # IV
            },
            'encrypted_content': core.OctetString(b'encrypted_dummy_content')
        }
    })
    
    print("✓ EnvelopedData structure created")
    print(f"Version: {enveloped_data['version'].native}")
    enc_alg = enveloped_data['encrypted_content_info']['content_encryption_algorithm']['algorithm'].native
    print(f"Encryption algorithm: {enc_alg}")
    content_alg = enveloped_data['encrypted_content_info']['content_encryption_algorithm']
    assert (content_alg.encryption_cipher, content_alg.key_length) == ('aes', 16)
    assert content_alg.encryption_iv == b'1234567890123456'
    
    # Test ContentInfo wrapper
    content_info = cms.ContentInfo({
        'content_type': 'signed_data',
        'content': signed_data
    })
    
    print("✓ ContentInfo wrapper created")
    print(f"Content type: {content_info['content_type'].native}")
    
    # Test encoding
    encoded_cms = content_info.dump()
    print(f"✓ CMS encoded: {len(encoded_cms)} bytes")
    
    # Parse back
    parsed_cms = cms.ContentInfo.load(encoded_cms)
    print("✓ CMS parsed successfully")
    print(f"Parsed content type: {parsed_cms['content_type'].native}")
    assert parsed_cms['content_type'].native == 'signed_data'
    parsed_signed = parsed_cms['content']
    assert parsed_signed['version'].native == 'v1'
    assert parsed_signed['digest_algorithms'][0]['algorithm'].native == 'sha256'
    assert parsed_signed['encap_content_info']['content'].native == b'This is the content to be signed'
    
except Exception as e:
    print(f"Error in CMS operations: {e}")
    raise AssertionError(
        f"asn1crypto example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed")